In [ ]:
from model import EmotionCNN
from xAI.gradcam import  overlay_heatmap, gradcam
from xAI.smoothGrad import coumpute_smoothGrad
from xAI.occlusion import occlusion_saliency
from xAI.LayerActivation import get_conv_layer, get_layer_activation, layer_activation_heatmap_from_tensor
import torch
import torch.nn as nn
import cv2
import numpy as np
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image




In [ ]:
idx_to_emotion = {
    0: "surprise",    
    1: "fear",        
    2: "disgust",    
    3: "happiness",   
    4: "sadness",     
    5: "anger",       
}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionCNN(num_classes=6).to(device)
WEIGHTS_PATH = "best_model_cosine.pt"
state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state)
model.eval()





In [ ]:
# load image
img = Image.open("evaluation/angry.jpg")



transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


tensor = transform(img).unsqueeze(0).to(device)



logits = model(tensor)
probs = torch.softmax(logits, dim=1)
conf, pred = torch.max(probs, dim=1)
pred_idx = int(pred.item())
conf= float(conf.item())
emotion = idx_to_emotion.get(pred_idx, str(pred_idx))
print("Emotion:", emotion, "Confidence:", conf)



img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


heatmap1 = gradcam(model, img_cv, pred_idx)
superimposed_img1 = overlay_heatmap(img_cv, heatmap1)
superimposed_img1 = cv2.cvtColor(superimposed_img1, cv2.COLOR_BGR2RGB)


heatmap2 = coumpute_smoothGrad(model, img_cv, pred_idx, 30)
superimposed_img2 = overlay_heatmap(img_cv, heatmap2)
superimposed_img2 = cv2.cvtColor(superimposed_img2, cv2.COLOR_BGR2RGB)

heatmap3 = occlusion_saliency(model, tensor, pred_idx)
superimposed_img3 = overlay_heatmap(img_cv, heatmap3.numpy())
superimposed_img3 = cv2.cvtColor(superimposed_img3, cv2.COLOR_BGR2RGB)


layer1 = get_conv_layer(model, which="last")
activation1 = get_layer_activation(model, layer1, tensor)
heatmap4 = layer_activation_heatmap_from_tensor(activation1)
heatmap4 = heatmap4.numpy()
superimposed_img4 = overlay_heatmap(img_cv, heatmap4)
superimposed_img4= cv2.cvtColor(superimposed_img4, cv2.COLOR_BGR2RGB)



plt.figure(figsize=(10,5))

plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img)
plt.text(
    0.5, -0.20,
    f"prediction: {emotion}\nConfidence: {conf:.2f}",
    ha="center",
    transform=plt.gca().transAxes
)
plt.axis('off')



plt.subplot(1,5,2)
plt.title("Grad-CAM")
plt.imshow(superimposed_img1)
plt.axis('off')

plt.subplot(1,5,3)
plt.title("SmoothGrad")
plt.imshow(superimposed_img2)
plt.axis('off')

plt.subplot(1,5,4)
plt.title("Occlusion")
plt.imshow(superimposed_img3)
plt.axis('off')

plt.subplot(1,5,5)
plt.title("LayerActivation")
plt.imshow(superimposed_img4)
plt.axis('off')

plt.show()






